# Monitoring and MLOps: worked solution

Worked solution for the exercise in the lesson **[Monitoring and MLOps](https://ramkumarmd.com/medical-ai/evaluation-safety/monitoring-and-mlops)** (*Evaluation, safety & deployment* track, Medical AI course by Dr. Ram Kumar G). Try the [exercise notebook](05-monitoring-and-mlops.ipynb) first. Other answers can be just as good: the written parts show one reasonable answer, not the only one.

**In short:** A clinical model's performance can change after go-live because patients, practice, coding or equipment change. Monitoring tracks the model's inputs (are they drifting?), its outputs, and, once outcomes are known, its calibration and discrimination, with alerts when they move outside agreed limits. MLOps is the engineering discipline that makes this routine: versioned models and data, reproducible pipelines, controlled updates and rollback. Regulators increasingly expect it, for example through change-control plans for AI devices.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Monitoring inputs and calibration over time

Twelve months of synthetic monitoring, with a lab assay change in month 7.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(11)
predictors = ["age", "egfr", "albuminuria"]

def month_of_patients(n, assay_bias=0.0):
    d = pd.DataFrame({"age": rng.normal(60, 10, n).round(),
                      "true_egfr": rng.normal(70, 20, n).clip(15, 120),
                      "albuminuria": rng.binomial(1, 0.3, n)})
    logit = -2.0 + 0.02 * (d.age - 60) - 0.04 * (d.true_egfr - 70) + 1.0 * d.albuminuria
    d["outcome"] = rng.binomial(1, 1 / (1 + np.exp(-logit)))
    d["egfr"] = (d.true_egfr + assay_bias).round()
    return d

development = month_of_patients(8000)
model = LogisticRegression(max_iter=1000).fit(development[predictors], development.outcome)

def psi(expected, actual, bins=10):
    """Population stability index: how far a variable's distribution has moved from baseline."""
    edges = np.quantile(expected, np.linspace(0, 1, bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    e = np.histogram(expected, edges)[0] / len(expected)
    a = np.histogram(actual, edges)[0] / len(actual)
    e, a = np.clip(e, 1e-6, None), np.clip(a, 1e-6, None)
    return float(np.sum((a - e) * np.log(a / e)))

print("month  eGFR PSI  observed  expected  O/E   alert")
for month in range(1, 13):
    d = month_of_patients(600, assay_bias=15 if month >= 7 else 0)   # new lab assay from month 7
    expected = model.predict_proba(d[predictors])[:, 1].mean()
    observed = d.outcome.mean()
    drift = psi(development.egfr, d.egfr)
    alerts = [a for a, bad in [("input drift", drift > 0.2),
                               ("calibration", not 0.8 <= observed / expected <= 1.25)] if bad]
    print(f"{month:>5}  {drift:8.2f}  {observed:8.1%}  {expected:8.1%}  "
          f"{observed / expected:4.2f}  {', '.join(alerts)}")

# month  eGFR PSI  observed  expected  O/E   alert
#     1      0.01     18.0%     19.8%  0.91
#     2      0.01     22.3%     19.1%  1.17
#     3      0.01     18.8%     19.3%  0.98
#     4      0.02     20.8%     19.3%  1.08
#     5      0.01     20.3%     19.2%  1.06
#     6      0.02     19.7%     17.5%  1.12
#     7      0.54     19.3%     12.3%  1.57  input drift, calibration
#     8      0.50     19.0%     12.7%  1.49  input drift, calibration
#     9      0.57     18.7%     12.4%  1.51  input drift, calibration
#    10      0.55     18.7%     12.1%  1.54  input drift, calibration
#    11      0.64     20.2%     11.7%  1.72  input drift, calibration
#    12      0.61     20.7%     11.8%  1.76  input drift, calibration

## Exercise

**Goal:** Build a monitoring dashboard for a synthetic deployed model and design the response to drift.

**Data:** Synthetic monthly data generated in the notebook (no real patient data) (Generated in the notebook)

### Step 1

Reproduce the 12-month monitoring table and plot PSI and O/E over time with the alert limits.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

def make_month(r, n, assay_bias=0.0, age_mean=60):
    d = pd.DataFrame({"age": r.normal(age_mean, 10, n).round(),
                      "true_egfr": r.normal(70, 20, n).clip(15, 120),
                      "albuminuria": r.binomial(1, 0.3, n)})
    logit = -2.0 + 0.02 * (d.age - 60) - 0.04 * (d.true_egfr - 70) + 1.0 * d.albuminuria
    d["outcome"] = r.binomial(1, 1 / (1 + np.exp(-logit)))
    d["egfr"] = (d.true_egfr + assay_bias).round()
    return d

def psi_binary(expected, actual):
    e = np.clip(np.array([1 - expected.mean(), expected.mean()]), 1e-6, None)
    a = np.clip(np.array([1 - actual.mean(), actual.mean()]), 1e-6, None)
    return float(np.sum((a - e) * np.log(a / e)))

def monitor(months, predict=None, seed=21):
    r = np.random.default_rng(seed)
    predict = predict or (lambda d: model.predict_proba(d[predictors])[:, 1])
    rows = []
    for m, settings in months:
        d = make_month(r, 600, **settings)
        p = predict(d)
        oe = d.outcome.mean() / p.mean()
        rows.append({"month": m, "PSI eGFR": psi(development.egfr, d.egfr), "PSI age": psi(development.age, d.age),
                     "PSI albuminuria": psi_binary(development.albuminuria, d.albuminuria),
                     "O/E": oe, "AUROC": roc_auc_score(d.outcome, p), "data": d})
    return pd.DataFrame(rows).set_index("month")

assay_change = [(m, {"assay_bias": 15 if m >= 7 else 0}) for m in range(1, 13)]
mon = monitor(assay_change)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
ax1.plot(mon.index, mon["PSI eGFR"], "o-"); ax1.axhline(0.2, ls="--", c="red"); ax1.set(title="eGFR PSI (alert above 0.2)", xlabel="month")
ax2.plot(mon.index, mon["O/E"], "o-"); ax2.axhspan(0.8, 1.25, color="green", alpha=0.1)
ax2.set(title="Observed / expected (acceptable 0.8-1.25)", xlabel="month")
plt.tight_layout(); plt.show()

### Step 2

Add PSI for age and albuminuria, and a monthly AUROC once outcomes are available.

In [ ]:
print(mon[["PSI eGFR", "PSI age", "PSI albuminuria", "O/E", "AUROC"]].round(3).to_string())

Only the eGFR distribution moves; age and albuminuria stay stable, which points straight at the cause (a
laboratory change, not a change in patients). AUROC hardly moves, because a constant offset in eGFR does not
change the ranking; calibration (O/E) is what breaks. Outcomes arrive late in practice (here, three years), so a
real dashboard would show AUROC and O/E with that delay, and rely on input drift for early warning.

### Step 3

Simulate a gradual case-mix change (age rising by one year each month) and see when the alerts fire.

In [ ]:
gradual = [(m, {"age_mean": 60 + m}) for m in range(1, 25)]
mon_age = monitor(gradual, seed=22)
flag = mon_age["PSI age"] > 0.2
first = flag.idxmax() if flag.any() else None
print(mon_age[["PSI age", "O/E", "AUROC"]].round(3).iloc[::3].to_string())
print(f"\nAge PSI first above 0.2 in month {first}; O/E outside 0.8-1.25 in months: "
      f"{list(mon_age.index[(mon_age['O/E'] < 0.8) | (mon_age['O/E'] > 1.25)])}")

A slow drift stays under the alert limit for five months (age PSI first exceeds 0.2 in month 6) while the
population changes steadily: each month looks
almost like the last. The model here stays calibrated because age is in the model and its effect has not
changed, so this drift is not harmful; input drift is a reason to look, not proof that the model is wrong.
Comparing with the original baseline (not last month) is what lets slow drift show up at all.

### Step 4

Recalibrate the model on months 7–9 and show monitoring for months 10–12 after the update.

In [ ]:
from sklearn.linear_model import LogisticRegression

recal_data = pd.concat(mon.loc[7:9, "data"].tolist())
lo = lambda d: np.log((q := model.predict_proba(d[predictors])[:, 1]) / (1 - q)).reshape(-1, 1)
recalibrator = LogisticRegression(penalty=None).fit(lo(recal_data), recal_data.outcome)
print(f"Recalibration on months 7-9 ({len(recal_data)} patients): slope {recalibrator.coef_[0][0]:.2f}, "
      f"intercept {recalibrator.intercept_[0]:.2f}")

updated = lambda d: recalibrator.predict_proba(lo(d))[:, 1]
after = monitor([(m, {"assay_bias": 15}) for m in range(10, 13)], predict=updated, seed=23)
before = monitor([(m, {"assay_bias": 15}) for m in range(10, 13)], seed=23)
print(pd.DataFrame({"O/E before update": before["O/E"], "O/E after update": after["O/E"],
                    "AUROC after": after["AUROC"]}).round(2).to_string())

After recalibration, O/E moves from about 1.5 to about 0.9, inside the acceptable range (slightly over-correcting,
because three months of outcomes are a noisy basis). The eGFR PSI alert would keep firing against the original baseline,
so after an approved update the baseline for input monitoring should be reset to the new assay (and the change
documented). Using outcomes from months 7-9 assumes outcomes are known quickly; with a three-year outcome, the
practical fix is to correct the eGFR offset between the assays.

### Step 5

Write a one-page monitoring plan: metrics, limits, owner, response times and how to switch the model off.

Example monitoring plan (kidney-risk model in clinic use):

| Metric | Frequency | Alert limit | Owner | Response time |
|---|---|---|---|---|
| Input drift (PSI) for each predictor, against the validation baseline | monthly | PSI > 0.2 | data scientist | review within 2 weeks |
| Missing-data rate per predictor | weekly | doubles from baseline | data engineer | 1 week |
| Calibration (O/E and calibration slope), once outcomes are available | quarterly | O/E outside 0.8-1.25 | clinical lead | review within 1 month |
| Discrimination (AUROC) with confidence interval | quarterly | below 0.70, or a fall of 0.05 | clinical lead | 1 month |
| Subgroup calibration (age, sex, eGFR band) | twice a year | O/E outside 0.8-1.25 in any group | clinical safety officer | 1 month |
| Upstream changes (lab assays, coding, EHR upgrades) | on notice | any change | IT change board | before go-live of the change |

**Switching off:** the clinical safety officer can withdraw the model at any time; it is suspended automatically
if inputs fail validation (for example a missing feed), and clinicians are told within one working day, with the
fallback process (usual clinical assessment) described in advance. Every alert, decision and model version is
logged, and an annual review decides whether to keep, update or retire the model.

---

Lesson: [https://ramkumarmd.com/medical-ai/evaluation-safety/monitoring-and-mlops](https://ramkumarmd.com/medical-ai/evaluation-safety/monitoring-and-mlops). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.